In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
cd /content/drive/MyDrive/t2dm/sheets

/content/drive/MyDrive/t2dm/sheets


In [ ]:
import pandas as pd

In [ ]:
workbook0 = pd.read_excel('AIG_ROC_Data_March.xlsx', sheet_name=None)
workbook1 = pd.read_excel('subject_updated_11.03.2026.xlsx', sheet_name=None)

In [ ]:
img = workbook1['Imaging data'][['Sub ID','PDFF', 'PDFF Head', 'PDFF Body', 'PDFF Tail']]

In [ ]:
infu = workbook1['Clamp data'][['Sub ID', 'first_phase_ISR', 'second_phase_ISR_10_120']]

In [ ]:
con = workbook0['Control'][['Sample ID', 'BMI (kg/m2)', 'Body weight (kg)']]
pd1 = workbook0['PD 1'][['Sample ID', 'BMI (kg/m2)', 'Body weight (kg)']]
pd2 = workbook0['PD 2'][['Sample ID', 'BMI (kg/m2)', 'Body weight (kg)']]

In [ ]:
all = pd.concat([con, pd1, pd2])

In [ ]:
all.shape

(243, 3)

In [ ]:
pd1.shape

(135, 3)

In [ ]:
workbook3 = pd.read_excel('For_correlation.xlsx', sheet_name=None)

In [ ]:
cor_all = pd.merge(workbook3['All'], all, left_on='Sub ID', right_on='Sample ID', how='left')
cor_ir = pd.merge(workbook3['IR'], pd2, left_on='Sub ID', right_on='Sample ID', how='left')
cor_id = pd.merge(workbook3['ID'], pd1, left_on='Sub ID', right_on='Sample ID', how='left')
cor_ct = pd.merge(workbook3['CT'], con, left_on='Sub ID', right_on='Sample ID', how='left')

TypeError: Can only merge Series or DataFrame objects, a <class 'builtin_function_or_method'> was passed

In [ ]:
cor_all = pd.merge(workbook3['All'], infu, on='Sub ID', how='left')
cor_ir = pd.merge(workbook3['IR'], infu, on='Sub ID', how='left')
cor_id = pd.merge(workbook3['ID'], infu, on='Sub ID', how='left')
cor_ct = pd.merge(workbook3['CT'], infu, on='Sub ID', how='left')

In [ ]:
with pd.ExcelWriter("/content/drive/MyDrive/t2dm/results/For_correlation.xlsx", engine="openpyxl") as writer:
    cor_all.to_excel(writer, sheet_name="All", index=False)
    cor_ir.to_excel(writer, sheet_name="IR", index=False)
    cor_id.to_excel(writer, sheet_name="ID", index=False)
    cor_ct.to_excel(writer, sheet_name="CT", index=False)

In [ ]:
r = [' (ml/min)', ' (mg/min)']
time = [f'Basal Infusion rate{r[0]}'] + [f'{i} min Infusion rate{r[0]}' for i in range(2,12,2)] + [f'{i} min Infusion rate{r[0]}' for i in range(15,125, 5)] + [f'Basal Infusion rate{r[1]}'] + [f'{i} min Infusion rate{r[1]}' for i in range(2,12,2)] + [f'{i} min Infusion rate{r[1]}' for i in range(15,125, 5)]

In [ ]:
colsappnd = ['Sub ID'] + time + ['Body weight (kg)', '1st phase glucose infusion rate (mg/min) (0-10min)', '1st phase glucose infusion rate (mg/min/kg) (0-10min)', '2nd phase glucose infusion rate (mg/min) (10-120min)',
             '2nd phase glucose infusion rate (mg/min/kg) (10-120min)', '1st phase insulin secretion (uU/ml) (0-10min)', '2nd phase insulin secretion (uU/ml)(10-120min)',
             '1st phase C-peptide response (ng/ml) (0-10min)', '2nd phase C-peptide response (ng/ml) (10-120 min)', '1st phase glucose (mg/dl) (0-10min)', '2nd phase glucose (mg/dl) (10-120min)',
             'ISI (M/I) (mg/kg/min/uU/ml)',  'AUC insulin 0-10min', 'Disposition Index ']

In [ ]:
appnd = workbook1['Clamp data'][colsappnd]

In [ ]:
appnd['Clamp Steady state Infusion rate (ml/min/kg)']  = (appnd[[f'{i} min Infusion rate{r[0]}' for i in range(100,125, 5)]].mean(axis=1)) / (appnd['Body weight (kg)'])
appnd['Clamp Steady state Infusion rate (mg/min/kg)']  = (appnd[[f'{i} min Infusion rate{r[1]}' for i in range(100,125, 5)]].mean(axis=1)) / (appnd['Body weight (kg)'])
appnd['Basal Infusion rate (mg/min/kg)'] = appnd['Basal Infusion rate (mg/min)'] / appnd['Body weight (kg)']

/tmp/ipykernel_447/3427063971.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  appnd['Clamp Steady state Infusion rate (ml/min/kg)']  = (appnd[[f'{i} min Infusion rate{r[0]}' for i in range(100,125, 5)]].mean(axis=1)) / (appnd['Body weight (kg)'])
/tmp/ipykernel_447/3427063971.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  appnd['Clamp Steady state Infusion rate (mg/min/kg)']  = (appnd[[f'{i} min Infusion rate{r[1]}' for i in range(100,125, 5)]].mean(axis=1)) / (appnd['Body weight (kg)'])
/tmp/ipyke

In [ ]:
workbook1['Clamp data'].shape

(245, 201)

In [ ]:
con = pd.merge(workbook0['Control'], appnd, left_on='Sample ID', right_on='Sub ID', how='left')
pd1 = pd.merge(workbook0['PD 1'], appnd, left_on='Sample ID', right_on='Sub ID', how='left')
pd2 = pd.merge(workbook0['PD 2'], appnd, left_on='Sample ID', right_on='Sub ID', how='left')

In [ ]:
with pd.ExcelWriter("/content/drive/MyDrive/t2dm/results/AIG_ROC_Data_March_apnd.xlsx", engine="openpyxl") as writer:
    con.to_excel(writer, sheet_name="Control", index=False)
    pd1.to_excel(writer, sheet_name="PD 1", index=False)
    pd2.to_excel(writer, sheet_name="PD 2", index=False)
    workbook0['Unmatched'].to_excel(writer, sheet_name="Unmatched", index=False)

In [ ]:
workbook2 = pd.read_excel('Analysis.xlsx')

In [ ]:
workbook2.columns

Index(['Unnamed: 0', 'Sample ID', 'Basal Glucose (mg/dl)',
       'Basal insulin (µU/ml)', 'Basal C-peptide (ng/ml)', 'HOMA-IR',
       'HOMA B%', 'HOMA-IR (C-Peptide)', 'HOMA B% (C-Peptide)', 'HOMA2-IR',
       'HOMA2 B%', 'HOMA2-S', 'HOMA2-IR (C-Peptide)', 'HOMA2 B% (C-Peptide)',
       'HOMA2-S (C-Peptide)', '20/C-Peptide*Glucose', 'C-Peptide Index',
       'QUICKI', 'Hepatic Clearance', 'ISI', 'Disposition Index',
       '1st phase insulin secretion (2-10min)',
       '2nd phase insulin secretion (100-120min)',
       '1st phase C-peptide response (2-10min)',
       '2nd phase C-peptide response (100-120 min)', 'Clamp based Clearance',
       '1st Phase Glucose Infusion Rate', '2nd Phase Glucose Infusion Rate',
       'Clamp-Steady State Infusion Rate', '1st phase glucose',
       '2nd phase glucose', 'Clamp Steady State Glucose', 'First Phase ISR',
       'Second Phase ISR'],
      dtype='object')

In [ ]:
workbook2 = workbook2.drop(columns=['ISI', 'Disposition Index', '1st phase insulin secretion (2-10min)', '2nd phase insulin secretion (100-120min)', '1st phase C-peptide response (2-10min)',
                                    '2nd phase C-peptide response (100-120 min)', '1st Phase Glucose Infusion Rate', '2nd Phase Glucose Infusion Rate', 'Clamp-Steady State Infusion Rate'])

In [ ]:
appnd = appnd[['Sub ID', 'ISI (M/I) (mg/kg/min/uU/ml)', 'Disposition Index ', 'Basal Infusion rate (mg/min)', 'Basal Infusion rate (mg/min/kg)', '1st phase glucose infusion rate (mg/min) (0-10min)',
               '1st phase glucose infusion rate (mg/min/kg) (0-10min)', '2nd phase glucose infusion rate (mg/min) (10-120min)', '2nd phase glucose infusion rate (mg/min/kg) (10-120min)',
               'Clamp Steady state Infusion rate (mg/min/kg)', 'Clamp Steady state Infusion rate (ml/min/kg)', '1st phase insulin secretion (uU/ml) (0-10min)', '2nd phase insulin secretion (uU/ml)(10-120min)',
               '1st phase C-peptide response (ng/ml) (0-10min)', '2nd phase C-peptide response (ng/ml) (10-120 min)', '1st phase glucose (mg/dl) (0-10min)', '2nd phase glucose (mg/dl) (10-120min)']]

In [ ]:
workbook2.shape

(242, 25)

In [ ]:
analysis1 = pd.merge(workbook2, appnd, left_on='Sample ID', right_on='Sub ID', how='left')
analysis1 = analysis1.drop(columns=['Sub ID'])
analysis1 = analysis1[['Sample ID', 'Basal Glucose (mg/dl)', 'Basal insulin (µU/ml)', 'Basal C-peptide (ng/ml)', 'HOMA-IR', 'HOMA B%', 'HOMA-IR (C-Peptide)', 'HOMA B% (C-Peptide)', 'HOMA2-IR',
                       'HOMA2 B%', 'HOMA2-S', 'HOMA2-IR (C-Peptide)', 'HOMA2 B% (C-Peptide)', 'HOMA2-S (C-Peptide)', '20/C-Peptide*Glucose', 'C-Peptide Index', 'QUICKI', 'Hepatic Clearance',
                       'ISI (M/I) (mg/kg/min/uU/ml)', 'Disposition Index ','1st phase insulin secretion (uU/ml) (0-10min)', '2nd phase insulin secretion (uU/ml)(10-120min)',
                       '1st phase C-peptide response (ng/ml) (0-10min)', '2nd phase C-peptide response (ng/ml) (10-120 min)', 'Clamp based Clearance', 'Basal Infusion rate (mg/min)', 'Basal Infusion rate (mg/min/kg)',
                       '1st phase glucose infusion rate (mg/min) (0-10min)', '1st phase glucose infusion rate (mg/min/kg) (0-10min)', '2nd phase glucose infusion rate (mg/min) (10-120min)',
                       '2nd phase glucose infusion rate (mg/min/kg) (10-120min)', 'Clamp Steady state Infusion rate (mg/min/kg)', 'Clamp Steady state Infusion rate (ml/min/kg)', '1st phase glucose',
                       '2nd phase glucose', 'Clamp Steady State Glucose', 'First Phase ISR', 'Second Phase ISR']]
analysis1.isnull().sum()

,0
Sample ID,0
Basal Glucose (mg/dl),0
Basal insulin (µU/ml),0
Basal C-peptide (ng/ml),0
HOMA-IR,0
HOMA B%,0
HOMA-IR (C-Peptide),1
HOMA B% (C-Peptide),1
HOMA2-IR,26
HOMA2 B%,26


In [ ]:
analysis1.to_excel("Analysis_1.xlsx", index=False)

# **For_Correlation**

In [ ]:
workbook3 = pd.read_excel('For_correlation.xlsx', sheet_name=None)

In [ ]:
workbook3.keys()

dict_keys(['All', 'IR', 'ID', 'CT'])

In [ ]:
workbook3['All'].columns

Index(['Sno', 'Group', 'Sub ID', 'Unnamed: 3', 'HbA1c',
       'Basal Glucose (mmol/l)', 'Basal insulin (µU/ml)', 'Basal HOMA IR',
       'Basal HOMA B %', 'Basal C-peptide (ng/ml)', 'Basal C-peptide (nmol/L)',
       'Beta-cell index-cpep', 'Hepatic clearance index', 'QUICKY',
       'C-peptide adequacy index', 'Basal Infusion rate', 'ISI (M/I) New',
       'iAUC insulin 0-10min', 'Unnamed: 18', 'Disposition Index ',
       '1st phase insulin secretion (0-10min)',
       '2nd phase insulin secretion (100-120min)',
       '1st phase C-peptide response (0-10min)',
       '2nd phase C-peptide response (100-120 min)',
       'Hepatic Insulin clearance', 'Clamp-Insulin clearance',
       '1st pahse glucose infusion rate', '2nd phase infusion rate',
       'Clamp-Steady state infusion rate', '1st phase glucose',
       '2nd phase glucose', 'Clamp-Steady state gluocse', 'Dextrose infused',
       'Pancreas volume T1', 'Pancreas volume T2', 'Hepatic fat MRI',
       'Pancreas relaxation time 

In [ ]:
cor_all = pd.merge(workbook3['All'], img, on='Sub ID', how='left')
cor_ir = pd.merge(workbook3['IR'], img, on='Sub ID', how='left')
cor_id = pd.merge(workbook3['ID'], img, on='Sub ID', how='left')
cor_ct = pd.merge(workbook3['CT'], img, on='Sub ID', how='left')

In [ ]:
with pd.ExcelWriter("/content/drive/MyDrive/t2dm/results/For_correlation.xlsx", engine="openpyxl") as writer:
    cor_all.to_excel(writer, sheet_name="All", index=False)
    cor_ir.to_excel(writer, sheet_name="IR", index=False)
    cor_id.to_excel(writer, sheet_name="ID", index=False)
    cor_ct.to_excel(writer, sheet_name="CT", index=False)

In [ ]:
cols_drop = ['Basal Infusion rate', 'ISI (M/I)', 'Disposition Index ',
       '1st phase insulin secretion (0-10min)',
       '2nd phase insulin secretion (100-120min)',
       '1st phase C-peptide response (0-10min)',
       '2nd phase C-peptide response (100-120 min)','1st pahse glucose infusion rate', '2nd phase infusion rate', 'Clamp-Steady state infusion rate']

for i in workbook3:
    workbook3[i] = workbook3[i].drop(columns=cols_drop)

In [ ]:
appnd = appnd[['Sub ID', 'Basal Infusion rate (mg/min/kg)', 'ISI (M/I) (mg/kg/min/uU/ml)', 'Disposition Index ', '1st phase glucose infusion rate (mg/min) (0-10min)', '1st phase glucose infusion rate (mg/min/kg) (0-10min)',
      '2nd phase glucose infusion rate (mg/min) (10-120min)', '2nd phase glucose infusion rate (mg/min/kg) (10-120min)', 'Clamp Steady state Infusion rate (mg/min/kg)',
      '1st phase insulin secretion (uU/ml) (0-10min)', '2nd phase insulin secretion (uU/ml)(10-120min)', '1st phase C-peptide response (ng/ml) (0-10min)', '2nd phase C-peptide response (ng/ml) (10-120 min)']]

In [ ]:
cols_order = ['Sno', 'Group', 'Sub ID', 'HbA1c',
       'Basal Glucose (mmol/l)', 'Basal insulin (µU/ml)', 'Basal HOMA IR',
       'Basal HOMA B %', 'Basal C-peptide (ng/ml)', 'Basal C-peptide (nmol/L)',
       'Beta-cell index-cpep', 'Hepatic clearance index', 'QUICKY',
       'C-peptide adequacy index', 'Basal Infusion rate (mg/min/kg)', 'ISI (M/I) (mg/kg/min/uU/ml)',
       'iAUC insulin 0-10min', 'Disposition Index ',
       '1st phase insulin secretion (uU/ml) (0-10min)', '2nd phase insulin secretion (uU/ml)(10-120min)',
       '1st phase C-peptide response (ng/ml) (0-10min)', '2nd phase C-peptide response (ng/ml) (10-120 min)',
       'Hepatic Insulin clearance', 'Clamp-Insulin clearance',
      '1st phase glucose infusion rate (mg/min) (0-10min)',
       '1st phase glucose infusion rate (mg/min/kg) (0-10min)', '2nd phase glucose infusion rate (mg/min) (10-120min)', '2nd phase glucose infusion rate (mg/min/kg) (10-120min)',
       'Clamp Steady state Infusion rate (mg/min/kg)', '1st phase glucose',
       '2nd phase glucose', 'Clamp-Steady state gluocse', 'Dextrose infused',
       'Pancreas volume T1', 'Pancreas volume T2', 'Hepatic fat MRI',
       'Pancreas relaxation time T2', 'Pancreas relaxation time T2 Head',
       'Pancreas relaxation time T2 Body', 'Pancreas relaxation time T2 Tail',
       'Pancreas Signal Intensity (PSI)', 'PSI HEAD', 'PSI BODY ', 'PSI TAIL']

In [ ]:
cor_all = pd.merge(workbook3['All'], appnd, on='Sub ID', how='left')
cor_ir = pd.merge(workbook3['IR'], appnd, on='Sub ID', how='left')
cor_id = pd.merge(workbook3['ID'], appnd, on='Sub ID', how='left')
cor_ct = pd.merge(workbook3['CT'], appnd, on='Sub ID', how='left')

In [ ]:
cor_all.to_excel("/content/drive/MyDrive/t2dm/results/For_correlation_1.xlsx", index=False)